# FoodScholar — Quickstart

Three cells from zero to a populated graph using your local corpus + pre-computed NER/NEL.

```
pip install -e '.[elastic,neo4j,ontology]'
# local services (matching the inline config below):
docker run -d -p 9200:9200 -e discovery.type=single-node \
    -e xpack.security.enabled=false elasticsearch:8.13.0
docker run -d -p 7687:7687 -p 7474:7474 \
    -e NEO4J_AUTH=neo4j/change-me neo4j:5
```

This notebook does **not** run GLiNER. Pre-computed annotations from
`data/foodscholar/ner/*.csv` are loaded and attached to the chunks — fast,
deterministic, no model downloads. The "Under the hood" appendix at the
bottom shows how to do the same end-to-end with GLiNER + HNSW when no
pre-computed output is on disk.

## 1. Configure

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from foodscholar import FoodScholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")

CORPUS_DIR = REPO_ROOT / "data" / "foodscholar" / "corpus"   # chunks_*.csv
NEL_DIR    = REPO_ROOT / "data" / "foodscholar" / "ner"      # nel_chunks_*.csv
FOODON_OWL = REPO_ROOT / "data" / "foodon.owl"

fs = FoodScholar.from_config({
    "corpus": {
        "chunks_path": str(CORPUS_DIR),
        "annotated_snapshot_path": str(REPO_ROOT / "data" / "annotated.parquet"),
    },
    "ontology": {
        "foodon_path": str(FOODON_OWL),
        "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
        "prefix_filter": ["FOODON:"],
    },
    "storage": {
        "chunk_store": {
            "backend": "elastic",
            "url": "http://localhost:9200",
            "index": "foodscholar_chunks",
        },
        "graph_store": {
            "backend": "neo4j",
            "url": "bolt://localhost:7687",
            "user": "neo4j",
            "password": "change-me",
        },
    },
})
fs.info()

{'foodscholar': '0.1.0',
 'config_hash': '2450d8c0d3958b65',
 'chunk_store': 'elastic',
 'graph_store': 'neo4j',
 'embedder': 'lazy(router:allenai/specter2_base,BAAI/bge-large-en-v1.5)',
 'llm': 'mock-llm-v0',
 'ontology': 'configured',
 'ner': 'gliner',
 'nel_backend': 'hnsw',
 'prompt_version': 'v1'}

## 2. Init stores & ingest

`fs.init()` creates the Elastic index and the Neo4j unique constraints — idempotent.

`fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR)` reads every `chunks_*.csv`, attaches the matching annotations from `nel_*.csv`, and upserts to Elastic. **No GLiNER, no HNSW, no embedding** — this is fast and works without the `[annotate]` extra installed.

Chunk-text embeddings are populated in a separate step (§3) so you can iterate on annotations without re-encoding.

In [ ]:
fs.init()
meta = fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR, ignore_source_types=["abstract"])
print(meta)

## 3. Embed (optional, for vector search)

Run this once chunks are in the store and you want kNN search to work. It builds the
production embedder — `SourceTypeRouter(SPECTER2 for abstracts, BGE-large for guides /
textbooks)` — lazily on the first call (~1.7 GB model load) and patches each chunk's
`embedding` + `embedding_model` fields in Elastic via a scoped `_update` (annotations untouched).

Default `only_missing=True` skips chunks that already carry a real vector, so re-runs after
adding new chunks only encode what's new.

Requires the `[annotate]` extra (`pip install -e '.[annotate]'`).

In [ ]:
meta = fs.embed()
print(meta)

## 4. Build & explore entities

Linked entities are first-class. `fs.build_entities()` walks the chunk store, dedupes every `EntityLink` by `ontology_id` (FOODON, CHEBI, GAZ, ...), aggregates `mention_count` / `chunk_count` / sample `chunk_ids`, enriches FoodOn ids with label + synonyms + ancestors from the loaded ontology, and writes everything to:

- **Elastic** — a dedicated `foodscholar_chunks_entities` index for fast lookup + search.
- **Neo4j** — `(:Entity)` nodes plus `(:Chunk)-[:MENTIONS {confidence, method}]->(:Entity)` edges.

Then `fs.entities` is the user-facing handle: `.list(prefix="FOODON")`, `.get(id)`, `.search("olive")`, `.chunks_for(id)`.

In [ ]:
meta = fs.build_entities()
print(meta)


In [20]:
print(f"\ntotal entities: {len(fs.entities)}")
# Top FoodOn entities by chunk_count.
for ent in fs.entities.list(k=8):
    print(f"  {ent.ontology_id:24s}  {ent.label!r:40s}  chunks={ent.chunk_count}")

# Lexical search over labels + synonyms.
print("\nsearch 'olive':")
for ent in fs.entities.search("olive", k=3):
    print(f"  {ent.ontology_id}  {ent.label}")

# Reverse lookup: chunks that mention a specific entity.
top = fs.entities.list(prefix="FOODON", k=1)
if top:
    print(f"\nchunks mentioning {top[0].ontology_id} ({top[0].label}):")
    for c in fs.entities.chunks_for(top[0].ontology_id, k=3):
        print(f"  {c.chunk_id}  {c.text[:80]}")

POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.151s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.012s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.009s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.004s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.002s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.007s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 durat


total entities: 3879
  FOODON:00005147           'food'                                    chunks=660
  ENVO:00002264             'food waste'                              chunks=4
  GAZ:00002637              'UK'                                      chunks=59
  ONS:1000010               'sustainable diets'                       chunks=306
  FOODON:00003375           'environmentally sustainable diet'        chunks=8
  ONS:1000021               'vegans'                                  chunks=56
  ONS:1000020               'vegetarian diet'                         chunks=123
  FOODON:03306429           'meat'                                    chunks=407

search 'olive':
  FOODON:03317509  olive
  FOODON:03301826  olive oil
  FOODON:03306633  virgin olive oil

chunks mentioning FOODON:00003339 (calories):
  f9b47e7a-3df8-4d67-970c-55d5d5403dc7  The flip side of the high-protein craze may be the highfiber approach to dieting
  4072818c-bb45-47e8-9341-036fc0e1b8e9  Very-low-calorie liqu

## 5. Inspect


In [21]:
# A handful of chunks with their attached mentions / foodon ids.
for c in fs.chunk_store.scan()[:3]:
    print(f"\n[{c.chunk_id}] {c.source_type} — {c.text[:80]}...")
    print(f"  mentions:   {[m.text for m in c.mentions[:6]]}")
    print(f"  foodon_ids: {c.foodon_ids[:6]}")
    other = sorted({ln.ontology_id for ln in c.entity_links if not ln.ontology_id.startswith('FOODON:')})
    print(f"  other_links: {other[:6]}")

# BM25 round-trip against Elastic.
hits = fs.chunk_store.search("olive oil", k=3, use_vector=False)
print("\n'olive oil' top 3:")
for h in hits:
    print(f"  {h.chunk_id}  {h.text[:80]}")

POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.052s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.050s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.055s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.053s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.048s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.049s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.052s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.044s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.042s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.040s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.043s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.046s]
POST


[1a8523b7-09ce-4d08-8afe-7922db143e03] textbook — Wasted food and drink, regardless of its source, is harmful to the environment b...
  mentions:   ['food', 'Household waste', '2018', 'UK']
  foodon_ids: ['FOODON:00005147']
  other_links: ['ENVO:00002264', 'GAZ:00002637']

[1b417dd6-5577-426d-9369-9e40a4737284] textbook — Given the broad definitions of sustainable diets, it is also important to consid...
  mentions:   ['sustainable diets', 'environmentally sustainable diet']
  foodon_ids: ['FOODON:00003375']
  other_links: ['ONS:1000010']

[f34e64f7-56ae-4b3d-8875-a256df8c07a2] textbook — As per the Fischer and Garnett definition 32  and modelling undertaken on UK die...
  mentions:   ['UK', 'sustainable diets', 'vegan diets', 'vegetarianism', 'veganism', 'meat']
  foodon_ids: ['FOODON:03306429', 'FOODON:00001015']
  other_links: ['ENVO:00003862', 'GAZ:00002637', 'ONS:1000010', 'ONS:1000020', 'ONS:1000021']

'olive oil' top 3:
  bc46ba7c-2832-48bc-bebd-9e8b0f1ff92c  Dietary Reference 

## 6. Visualize

`fs.viz` builds typed graphs (`VizGraph`) at five abstraction levels and renders
them via pluggable backends. Requires the `[viz]` extra (`pip install -e '.[viz]'`).

| Level | Method                                  | What it shows |
| ----- | --------------------------------------- | ------------- |
| L0    | `fs.viz.entity_histogram(prefix=, k=)`  | Top-k entities by chunk_count (stats) |
| L1    | `fs.viz.entity_neighborhood(ontology_id)` | One entity + mentioning chunks + co-entities |
| L2    | `fs.viz.shelf(shelf_id)`                | One Layer A shelf + themes + chunks |
| L3    | `fs.viz.backbone(facet=)`               | Whole Layer A/B/C backbone |
| L4    | `fs.viz.ontology_subtree(ontology_id)`  | FoodOn ancestors + descendants |

Renderer choice via `.render(backend, output=...)`:
- `"pyvis"`      → interactive HTML (best in-notebook).
- `"cytoscape"`  → self-contained Cytoscape.js HTML, no Python deps beyond stdlib.
- `"graphviz"`   → static SVG / PNG (needs `dot` binary).
- `"matplotlib"` → bars / heatmaps over node weights (ignores edges; perfect for L0).

In [ ]:
# L0 — top FoodOn entities, rendered with matplotlib (inline figure).
fig = fs.viz.entity_histogram(prefix="FOODON", k=20).render("matplotlib")
fig

In [ ]:
# L1 — entity neighborhood. Cytoscape ships self-contained HTML; embed inline.
# Helper: embed any HTML *content* (not a path) inside the notebook so it
# survives Jupyter / JupyterLab / VS Code's different static-file routes.
# Uses an iframe `srcdoc`, which sandboxes the embedded JS/CSS from the page.
from IPython.display import HTML
from html import escape

def _embed(html: str, *, height: int = 600) -> HTML:
    return HTML(
        f'<iframe srcdoc="{escape(html, quote=True)}" '
        f'width="100%" height="{height}" '
        f'style="border:1px solid #E5E7EB;border-radius:6px;"></iframe>'
    )

top = fs.entities.list(prefix="FOODON", k=1)
anchor_id = top[0].ontology_id if top else "FOODON:03309927"

html = fs.viz.entity_neighborhood(anchor_id, max_chunks=8).render("cytoscape")
_embed(html, height=620)

In [ ]:
# L4 — ontology subtree around a single FoodOn id (pyvis interactive).
# Helper: embed any HTML *content* (not a path) inside the notebook so it
# survives Jupyter / JupyterLab / VS Code's different static-file routes.
# Uses an iframe `srcdoc`, which sandboxes the embedded JS/CSS from the page.
from IPython.display import HTML
from html import escape

def _embed(html: str, *, height: int = 600) -> HTML:
    return HTML(
        f'<iframe srcdoc="{escape(html, quote=True)}" '
        f'width="100%" height="{height}" '
        f'style="border:1px solid #E5E7EB;border-radius:6px;"></iframe>'
    )

import importlib.util as _u
if not _u.find_spec("pyvis"):
    display(HTML('<p><em>pyvis not installed — <code>pip install -e ".[viz]"</code></em></p>'))
else:
    # pyvis returns a full HTML document; iframe-srcdoc sandboxes it.
    html = fs.viz.ontology_subtree(anchor_id, max_descendants=20).render("pyvis")
    display(_embed(html, height=620))

In [ ]:
# L2 / L3 — shelves + themes. Empty-state until Layer A lands; render inline.
# Helper: embed any HTML *content* (not a path) inside the notebook so it
# survives Jupyter / JupyterLab / VS Code's different static-file routes.
# Uses an iframe `srcdoc`, which sandboxes the embedded JS/CSS from the page.
from IPython.display import HTML
from html import escape

def _embed(html: str, *, height: int = 600) -> HTML:
    return HTML(
        f'<iframe srcdoc="{escape(html, quote=True)}" '
        f'width="100%" height="{height}" '
        f'style="border:1px solid #E5E7EB;border-radius:6px;"></iframe>'
    )

rg_backbone = fs.viz.backbone()
print(rg_backbone)
html = rg_backbone.render("cytoscape")
_embed(html, height=400)

---

## Under the hood (appendix)

The three cells above are everything an end-user needs. The appendix below is
for contributors and shows the same pipeline **without** pre-computed NER/NEL —
i.e. running GLiNER + HNSW from scratch. Skip it unless you're debugging the
NER side.

### A1. Run GLiNER + HNSW directly

Requires `pip install -e '.[annotate]'`. First call downloads the GLiNER bio
model (~1.5 GB) and the BioLORD encoder (~440 MB), then builds and caches the
FoodOn HNSW index under `data/`. Subsequent runs are fast.

In [ ]:
import importlib.util as _u

needs = [pkg for pkg in ("gliner", "hnswlib", "sentence_transformers") if not _u.find_spec(pkg)]
if needs:
    print("Skipping — missing:", needs, "\nInstall with:  pip install -e '.[annotate]'")
else:
    fs_g = FoodScholar.from_config({
        "corpus": {"chunks_path": str(CORPUS_DIR)},
        "ontology": {
            "foodon_path": str(FOODON_OWL),
            "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
            "prefix_filter": ["FOODON:"],
        },
        "storage": {
            "chunk_store": {"backend": "memory"},
            "graph_store": {"backend": "memory"},
        },
    })
    one_file = next(iter(sorted(CORPUS_DIR.glob("*.csv"))))
    fs_g.ingest(one_file)  # no nel_dir → GLiNER + HNSW path
    c = fs_g.chunk_store.scan()[0]
    print(c.chunk_id, "mentions:", [m.text for m in c.mentions[:6]])

### A2. Inspect the graph

Layer A/B/C builders are still stubs — this appendix shows the `fs.graph` API
the builders will write through. Run it against the in-memory facade so it
doesn't depend on the live services.

In [ ]:
fs_local = FoodScholar.in_memory()
fs_local.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                         facet="dietary_patterns", depth=0)
fs_local.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                         shelf_ids=["s-med"], discovered_by="leiden",
                         discovery_version="nb")
shelf = fs_local.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("themes:", [t.label for t in shelf.themes()])